In [56]:
import os
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

os.environ["TYPESAFE_BASE_URL"] = "http://localhost:11434"
os.environ["TYPESAFE_API_KEY"] = "ollama"
os.environ["TYPESAFE_DEFAULT_MODEL"] = "nimble"

In [2]:
import pandas as pd

df = pd.read_csv("temperaturas.csv")
df.head()

,id,fecha,hora,temperatura_celsius
0,1,2026-10-15,08:00,18.5
1,2,2026-10-15,09:00,19.2
2,3,2026-10-15,10:00,21.0
3,4,2026-10-15,11:00,22.8
4,5,2026-10-15,12:00,24.5


In [3]:
tabla_temperaturas = df.to_dict(orient="records")
tabla_temperaturas

[{'id': 1,
  'fecha': '2026-10-15',
  'hora': '08:00',
  'temperatura_celsius': 18.5},
 {'id': 2,
  'fecha': '2026-10-15',
  'hora': '09:00',
  'temperatura_celsius': 19.2},
 {'id': 3,
  'fecha': '2026-10-15',
  'hora': '10:00',
  'temperatura_celsius': 21.0},
 {'id': 4,
  'fecha': '2026-10-15',
  'hora': '11:00',
  'temperatura_celsius': 22.8},
 {'id': 5,
  'fecha': '2026-10-15',
  'hora': '12:00',
  'temperatura_celsius': 24.5},
 {'id': 6,
  'fecha': '2026-10-15',
  'hora': '13:00',
  'temperatura_celsius': 25.1},
 {'id': 7,
  'fecha': '2026-10-15',
  'hora': '14:00',
  'temperatura_celsius': 24.8},
 {'id': 8,
  'fecha': '2026-10-15',
  'hora': '15:00',
  'temperatura_celsius': 23.6},
 {'id': 9,
  'fecha': '2026-10-15',
  'hora': '16:00',
  'temperatura_celsius': 22.0},
 {'id': 10,
  'fecha': '2026-10-15',
  'hora': '17:00',
  'temperatura_celsius': 20.4}]

In [4]:
#state_data = {
#    "registro_temperaturas": tabla_temperaturas
#}

## Test 1

Respuesta en base a la intención del query

In [57]:
from langchain_core.runnables import RunnableLambda

In [6]:
def nimble_router_node(inputs: dict) -> str:
    """Nodo que evalúa la consulta usando nimble y el dataset de temperaturas."""
    prompt = inputs.get("input", "")

    # Combinamos la tabla de datos y la pregunta del usuario dentro del 'state'
    state_data = {
        "registro_temperaturas": tabla_temperaturas,
        "consulta_usuario": prompt,
    }

    questions = {
        "intent": Choice(
            instructions=(
                "Basándote en la tabla de temperaturas y la consulta del"
                " usuario, clasifica el tipo de pregunta."
            ),
            criteria={
                "temp_max": (
                    "Preguntas sobre valores máximos, picos de calor o si se"
                    " superó cierta temperatura."
                ),
                "rango_horario": (
                    "Preguntas sobre horas específicas o momentos del día"
                    " (mañana, tarde, etc.)."
                ),
                "otro": "Preguntas generales que no se relacionan con los datos de temperatura.",
            },
        )
    }

    with TypeSafeClient() as client:
        res = client.system_one(state=state_data, questions=questions)

    return res.choices["intent"].choice


In [9]:

# 2. Crear el runnable de LangChain
router_runnable = RunnableLambda(nimble_router_node)

# 3. Probar la decisión
decision = router_runnable.invoke(
    {"input": "¿los 24 grados en algún momento del día?"}
)
print("Ruta seleccionada por LangChain:", decision)  # Devuelve 'temp_max'

Ruta seleccionada por LangChain: rango_horario


## Test 2

In [58]:
from typesafe_sdk import TypeSafeClient, Choice

def extraer_intent_con_nimble(pregunta: str) -> str:
    questions = {
        "metric": Choice(
            instructions="¿Qué métrica se está consultando?",
            criteria={
                "temperature": "Mediciones de temperatura o calor.",
                "humidity": "Niveles de humedad.",
                "pressure": "Presión atmosférica o de gas."
            }
        ),
        "aggregation": Choice(
            instructions="¿Qué tipo de cálculo solicita el usuario?",
            criteria={
                "MAX": "Pide el valor máximo, pico o más alto.",
                "MIN": "Pide el valor mínimo o más bajo.",
                "AVG": "Pide el promedio o media.",
                "NONE": "Pide ver todos los datos o la lista completa."
            }
        )
    }

    with TypeSafeClient() as client:
        res = client.system_one(state=pregunta, questions=questions)
        
    return res.choices["metric"].choice, res.choices["aggregation"].choice

# Ejemplo de uso directo
metrica, agregacion = extraer_intent_con_nimble("¿Cuál fue la temperatura máxima hoy?")
print(f"Métrica: {metrica}, Agregación: {agregacion}")

Métrica: temperature, Agregación: MAX


## test 3

In [59]:
import numpy as np

In [60]:
data = pd.DataFrame({'t':[1,2,3,45,67,34,12,112], 'h': [34,56,55,33,22,11,67,5]})

In [61]:
def fun_questions(pregunta):
    questions = {
        "variable": Choice(
            instructions ="¿De que variable esta hablando?",
            criteria={
                "t": "Si habla sobre la temperatura",
                "h": "si habla de la humedad",
                "th": "Si habla de la temperatura y humedad"
            }
        ),
        "estadistica": Choice(
            instructions="¿Qué tipo de cálculo solicita el usuario?",
            criteria={
                "MAX": "Pide el valor máximo, pico o más alto.",
                "MIN": "Pide el valor mínimo o más bajo.",
                "NP.MEAN": "Pide el promedio o media.",
                "NONE": "Pide ver todos los datos o la lista completa."
            }
        )
    }

    with TypeSafeClient() as client:
        res = client.system_one(state=pregunta, questions=questions)
        
    return res.choices["variable"].choice, res.choices["estadistica"].choice


In [62]:
query_text = input()

 cual la data mas pequeña obtenida de la variable asociada a vapor de agua en el ambiente?


In [63]:
variab, estadis = fun_questions(query_text) #"¿Cuál fue la temperatura máxima hoy?")
estadis_std = estadis.lower()

print(f"Variable: {variab} \nestadistico: {estadis}  \nvalor: {eval(estadis_std)(data[variab])}")

Variable: h 
estadistico: MIN  
valor: 5
